# Architecture comparison under the frozen primary protocol

This notebook compares three model families on exactly the same validated 5-second, 100 Hz, three-channel acceleration-magnitude windows and participant-level folds:

1. the already executed compact 1D CNN pilot;
2. a small InceptionTime-style multi-scale CNN;
3. MiniROCKET with a ridge classifier.

The comparison is a research benchmark, not a clinical model-selection claim. Evaluation remains participant-level, with fold-specific normalization and cross-dataset results retained as the most important generalization check.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import RidgeClassifier
import joblib
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name.lower() == 'archive':
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
INTERIM = PROJECT_ROOT / 'data' / 'interim'
MAG_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'
SPLITS_PATH = INTERIM / 'participant_splits.csv'
FOLDS = [0, 1, 2, 3, 4]
EPOCHS = 4
BATCH_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
metadata = pd.read_csv(METADATA_PATH)
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
splits = pd.read_csv(SPLITS_PATH)
magnitude_windows = np.load(MAG_PATH, mmap_mode='r')
assert magnitude_windows.shape == (len(metadata), 500, 3)
print('Magnitude windows:', magnitude_windows.shape)
print('Participants:', metadata['participant_key'].nunique())
print('Device:', DEVICE)

Magnitude windows: (18511, 500, 3)
Participants: 284
Device: cuda


In [2]:
def fold_roles(fold):
    role_map = splits[splits['fold'].eq(fold)].set_index('participant_key')['role']
    return metadata['participant_key'].map(role_map)

def fold_normalization(train_indices):
    total = np.zeros(3, dtype=np.float64)
    total_sq = np.zeros(3, dtype=np.float64)
    count = 0
    for start in range(0, len(train_indices), 512):
        batch = np.asarray(magnitude_windows[train_indices[start:start + 512]], dtype='float32')
        total += batch.sum(axis=(0, 1))
        total_sq += np.square(batch).sum(axis=(0, 1))
        count += batch.shape[0] * batch.shape[1]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8))
    return mean.astype('float32'), std.astype('float32')

def participant_class_weights(indices):
    frame = metadata.iloc[indices]
    participant_counts = frame.groupby('participant_key').size()
    class_counts = frame.groupby('label_binary').size()
    weights = frame['participant_key'].map(1.0 / participant_counts).to_numpy()
    weights *= frame['label_binary'].map(len(indices) / (2.0 * class_counts)).to_numpy()
    return (weights / weights.mean()).astype('float32')

class MagnitudeDataset(Dataset):
    def __init__(self, indices, mean, std, weights=None):
        self.indices = np.asarray(indices, dtype='int64')
        self.mean = mean.reshape(1, 3)
        self.std = std.reshape(1, 3)
        self.weights = np.ones(len(self.indices), dtype='float32') if weights is None else weights
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item])
        signal = ((np.asarray(magnitude_windows[index], dtype='float32') - self.mean) / self.std).T.copy()
        return torch.from_numpy(signal), torch.tensor(float(metadata.iloc[index]['label_binary'])), torch.tensor(float(self.weights[item])), torch.tensor(index)

def participant_metrics(indices, probabilities):
    frame = metadata.iloc[np.asarray(indices)].copy()
    frame['probability'] = probabilities
    participant = frame.groupby(['participant_key', 'dataset_id', 'label_binary'], as_index=False)['probability'].mean()
    y_true = participant['label_binary'].to_numpy()
    y_prob = participant['probability'].to_numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'participants': len(participant),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'f1': f1_score(y_true, y_pred),
    }, participant

In [3]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=16):
        super().__init__()
        bottleneck = min(32, in_channels)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck, kernel_size=1, bias=False)
        self.branches = nn.ModuleList([
            nn.Conv1d(bottleneck, out_channels, kernel_size=7, padding=3, bias=False),
            nn.Conv1d(bottleneck, out_channels, kernel_size=15, padding=7, bias=False),
            nn.Conv1d(bottleneck, out_channels, kernel_size=25, padding=12, bias=False),
        ])
        self.pool_branch = nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(out_channels * 4)
        self.activation = nn.GELU()
        self.residual = nn.Conv1d(in_channels, out_channels * 4, kernel_size=1, bias=False) if in_channels != out_channels * 4 else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x)
        branches = [branch(z) for branch in self.branches]
        branches.append(self.pool_branch(nn.functional.max_pool1d(x, kernel_size=3, stride=1, padding=1)))
        out = self.bn(torch.cat(branches, dim=1))
        return self.activation(out + self.residual(x))

class InceptionGaitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(InceptionBlock(3, 16), nn.MaxPool1d(2), InceptionBlock(64, 16), nn.AdaptiveAvgPool1d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 1))
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

class CompactGaitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=9, padding=4), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.GELU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(128, 1))
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

def train_torch_fold(fold, model_factory, model_name):
    roles = fold_roles(fold)
    train_indices = np.flatnonzero(roles.eq('training').to_numpy())
    validation_indices = np.flatnonzero(roles.eq('validation').to_numpy())
    mean, std = fold_normalization(train_indices)
    weights = participant_class_weights(train_indices)
    train_loader = DataLoader(MagnitudeDataset(train_indices, mean, std, weights), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(MagnitudeDataset(validation_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = model_factory().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auc = -np.inf
    best_state = None
    patience = 2
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for signals, labels, batch_weights, _ in train_loader:
            optimizer.zero_grad()
            logits = model(signals.to(DEVICE))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, labels.to(DEVICE), reduction='none') * batch_weights.to(DEVICE)).mean()
            loss.backward()
            optimizer.step()
        model.eval()
        val_probabilities = []
        with torch.no_grad():
            for signals, _, _, _ in validation_loader:
                val_probabilities.extend(torch.sigmoid(model(signals.to(DEVICE))).detach().cpu().numpy())
        metrics, _ = participant_metrics(validation_indices, np.asarray(val_probabilities))
        print(f"{model_name} fold={fold} epoch={epoch} auc={metrics['roc_auc']:.3f} bal_acc={metrics['balanced_accuracy']:.3f}")
        if metrics['roc_auc'] > best_auc:
            best_auc = metrics['roc_auc']
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience = 2
        else:
            patience -= 1
            if patience == 0:
                break
    model.load_state_dict(best_state)
    model.eval()
    val_probabilities = []
    with torch.no_grad():
        for signals, _, _, _ in validation_loader:
            val_probabilities.extend(torch.sigmoid(model(signals.to(DEVICE))).detach().cpu().numpy())
    metrics, participant = participant_metrics(validation_indices, np.asarray(val_probabilities))
    metrics.update({'model': model_name, 'fold': fold})
    participant['model'] = model_name
    participant['fold'] = fold
    return metrics, participant

In [4]:
inception_results = []
inception_predictions = []
compact_gpu_results = []
compact_gpu_predictions = []
for fold in FOLDS:
    metrics, participant = train_torch_fold(fold, InceptionGaitCNN, 'inception_cnn')
    inception_results.append(metrics)
    inception_predictions.append(participant)
    metrics, participant = train_torch_fold(fold, CompactGaitCNN, 'compact_cnn_gpu')
    compact_gpu_results.append(metrics)
    compact_gpu_predictions.append(participant)

inception_results = pd.DataFrame(inception_results)
inception_predictions = pd.concat(inception_predictions, ignore_index=True)
compact_gpu_results = pd.DataFrame(compact_gpu_results)
compact_gpu_predictions = pd.concat(compact_gpu_predictions, ignore_index=True)
print(inception_results.round(3).to_string(index=False))

inception_cnn fold=0 epoch=1 auc=0.907 bal_acc=0.790


inception_cnn fold=0 epoch=2 auc=0.929 bal_acc=0.817


inception_cnn fold=0 epoch=3 auc=0.922 bal_acc=0.831


inception_cnn fold=0 epoch=4 auc=0.929 bal_acc=0.841


compact_cnn_gpu fold=0 epoch=1 auc=0.910 bal_acc=0.730


...

inception_cnn fold=4 epoch=3 auc=0.995 bal_acc=0.900


inception_cnn fold=4 epoch=4 auc=0.993 bal_acc=0.943


compact_cnn_gpu fold=4 epoch=1 auc=0.993 bal_acc=0.857


compact_cnn_gpu fold=4 epoch=2 auc=0.985 bal_acc=0.914


compact_cnn_gpu fold=4 epoch=3 auc=0.993 bal_acc=0.857
 participants  balanced_accuracy  roc_auc    f1         model  fold
           57              0.817    0.929 0.836 inception_cnn     0
           58              0.848    0.938 0.870 inception_cnn     1
           57              0.875    0.976 0.857 inception_cnn     2
           56              0.924    0.973 0.943 inception_cnn     3
           56              0.900    0.995 0.889 inception_cnn     4


In [5]:
rocket_results = []
rocket_predictions = []
for fold in FOLDS:
    roles = fold_roles(fold)
    train_indices = np.flatnonzero(roles.eq('training').to_numpy())
    validation_indices = np.flatnonzero(roles.eq('validation').to_numpy())
    mean, std = fold_normalization(train_indices)
    train_x = ((np.asarray(magnitude_windows[train_indices], dtype='float32') - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)).transpose(0, 2, 1)
    validation_x = ((np.asarray(magnitude_windows[validation_indices], dtype='float32') - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)).transpose(0, 2, 1)
    train_y = metadata.iloc[train_indices]['label_binary'].to_numpy()
    weights = participant_class_weights(train_indices)
    rocket = MiniRocketMultivariate(num_kernels=2000, max_dilations_per_kernel=16, n_jobs=1, random_state=42)
    train_features = rocket.fit_transform(train_x)
    validation_features = rocket.transform(validation_x)
    classifier = RidgeClassifier(alpha=1.0)
    classifier.fit(train_features, train_y, sample_weight=weights)
    joblib.dump({'transformer': rocket, 'classifier': classifier, 'mean': mean, 'std': std, 'fold': fold, 'seed': 42, 'window_seconds': 5.0, 'channels': ['LB', 'LF', 'RF'], 'feature_shape': train_features.shape[1]}, PROCESSED / f'minirocket_ridge_fold_{fold}_seed_42.joblib', compress=3)
    decision = classifier.decision_function(validation_features)
    probabilities = 1.0 / (1.0 + np.exp(-np.clip(decision, -30, 30)))
    metrics, participant = participant_metrics(validation_indices, probabilities)
    metrics.update({'model': 'minirocket_ridge', 'fold': fold})
    participant['model'] = 'minirocket_ridge'
    participant['fold'] = fold
    rocket_results.append(metrics)
    rocket_predictions.append(participant)
    print(f"rocket fold={fold} auc={metrics['roc_auc']:.3f} bal_acc={metrics['balanced_accuracy']:.3f}")

rocket_results = pd.DataFrame(rocket_results)
rocket_predictions = pd.concat(rocket_predictions, ignore_index=True)

rocket fold=0 auc=0.955 bal_acc=0.901


rocket fold=1 auc=0.956 bal_acc=0.955


rocket fold=2 auc=0.996 bal_acc=0.962


rocket fold=3 auc=0.959 bal_acc=0.900


rocket fold=4 auc=0.995 bal_acc=0.962


In [6]:
compact_results = compact_gpu_results.rename(columns={'model': 'model'})
compact_results = compact_results[['model', 'fold', 'participants', 'balanced_accuracy', 'roc_auc', 'f1']]
all_results = pd.concat([compact_results, inception_results[['model', 'fold', 'participants', 'balanced_accuracy', 'roc_auc', 'f1']], rocket_results[['model', 'fold', 'participants', 'balanced_accuracy', 'roc_auc', 'f1']]], ignore_index=True)
summary = all_results.groupby('model', as_index=False).agg(
    folds=('fold', 'nunique'),
    mean_balanced_accuracy=('balanced_accuracy', 'mean'),
    sd_balanced_accuracy=('balanced_accuracy', 'std'),
    mean_roc_auc=('roc_auc', 'mean'),
    sd_roc_auc=('roc_auc', 'std'),
    mean_f1=('f1', 'mean'),
)
print(summary.round(3).to_string(index=False))
all_results.to_csv(PROCESSED / 'architecture_comparison_fold_results.csv', index=False)
summary.to_csv(PROCESSED / 'architecture_comparison_summary.csv', index=False)
pd.concat([compact_gpu_predictions, inception_predictions, rocket_predictions], ignore_index=True).to_csv(PROCESSED / 'architecture_comparison_participant_predictions.csv', index=False)

           model  folds  mean_balanced_accuracy  sd_balanced_accuracy  mean_roc_auc  sd_roc_auc  mean_f1
 compact_cnn_gpu      5                   0.857                 0.079         0.973       0.024    0.845
   inception_cnn      5                   0.873                 0.042         0.962       0.028    0.879
minirocket_ridge      5                   0.936                 0.033         0.972       0.021    0.956


## Selection rule

Select the next model using mean participant-level AUROC together with cross-dataset performance, fold variability and the non-CVA specificity stress test. If two models are practically equivalent, prefer the simpler or more interpretable model. A high within-dataset score alone is not sufficient.